## Clean & Analyze X Social Media Data - Jamaica Gleaner

### Background Info:
#### Social Media Platform: X.com
#### Company: Jamaica Gleaner
#### Type of Company: Multi-media organization (newspaper, radio etc.)
#### No. of Tweets Analyzed: ---

### Project Setup

In [1]:
# Import Libraries
import tweepy
import pandas as pd
import matplotlib.pyplot as plt
import re

from dotenv import load_dotenv
import os

from textblob import TextBlob

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.cluster import KMeans

In [2]:
# Load Environment Variables
load_dotenv()

True

In [3]:
# Retrieves Environment Variables
# CONSUMER_KEY = os.getenv("CONSUMER_KEY")
# SECRET_KEY = os.getenv("SECRET_KEY")
BEARER_TOKEN = os.getenv("BEARER_TOKEN")

In [4]:
# Verify Variables Were Loaded
# print(CONSUMER_KEY is not None)
# print(SECRET_KEY is not None)
print(BEARER_TOKEN is not None)

True


In [5]:
# Authenticates with X API
client = tweepy.Client(
    bearer_token=BEARER_TOKEN,
    wait_on_rate_limit=True # automatically pauses program if API rate limit reached; resumes when requests allowed again
)

### Retrieve Company's User ID

In [6]:
# Retrieves Company's User ID
username = "JamaicaGleaner"

In [7]:
# Retrieves Numeric User ID
user = client.get_user(username=username)

user_id = user.data.id

print(user_id)

HTTPException: 402 Payment Required
credits depleted

### Retrieve Posts Via Pagination

In [14]:
# Stores Post Data
posts_data = [] # stores posts retrieved

pagination_token = None # keeps track of next page of results

# Retrieve Posts Via Pagination Loop
while True:
    response = client.get_users_tweets(
        id=user_id,

        max_results=100,
    
        tweet_fields=[
            "created_at",
            "public_metrics",
            "attachments"
        ],
    
        expansions=[
            "attachments.media_keys"
        ],
    
        media_fields=[
            "type",
             "url",
            "preview_image_url",
            "duration_ms",
            "width",
            "height",
            "alt_text"
        ],
    
        pagination_token=pagination_token
    )

    # Media Look Up Dictionary
    media_lookup = {}

    if (response.includes is not None and "media" in response.includes):

        for media in response.includes["media"]:

            media_lookup[media.media_key] = media

    # Extracts Posts 
    if response.data is not None:
    
        for tweet in response.data:
    
            metrics = tweet.public_metrics
            
            media_type = None
            
            media_url = None
        
            if (
                tweet.attachments is not None and
                "media_keys" in tweet.attachments
            ):
        
                key = tweet.attachments["media_keys"][0]
        
                if key in media_lookup:
            
                    media = media_lookup[key]
            
                    media_type = media.type
            
                    media_url = getattr(media, "url", None)
        
            posts_data.append({
            
                "tweet_id": tweet.id,
                
                "text": tweet.text,
                
                "created_at": tweet.created_at,
            
                "likes": metrics["like_count"],
                
                "replies": metrics["reply_count"],
                
                "reposts": metrics["retweet_count"],
                
                "quotes": metrics["quote_count"],
                
                "media_type": media_type,
                
                "media_url": media_url
            })

    # Checks if another page of posts exists
    if (
        response.meta is not None
        and "next_token" in response.meta
    ):

        pagination_token = response.meta["next_token"]

    else:

        break

# Converts to Dataframe
df = pd.DataFrame(posts_data)

# Creates column indicating whether post includes media
df["contains_media"] = df["media_type"].notna()

# Displays first 5 rows + no. of rows & columns
df.head()
df.shape()

# Display media-related columns
print(
    df[
        [
            "text",
            "media_type",
            "media_url",
            "contains_media"
        ]
    ].head()
)

# Display the total number of posts retrieved
print(f"\nTotal Posts Retrieved: {len(df)}")

HTTPException: 402 Payment Required
credits depleted

### Preprocess Post Text 

In [ ]:
# Import Libraries
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
# Download NLTK resources
nltk.download("stopwords")

nltk.download("wordnet")

nltk.download("omw-1.4")

In [ ]:
# Create Stopword List and Lemmatizer
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

In [ ]:
# Create Preprocessing Function
def preprocess_text(text):

    # Convert missing values to an empty string
    if pd.isna(text):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        "",
        text
    )

    # Remove @mentions
    text = re.sub(
        r"@\w+",
        "",
        text
    )

    # Remove hashtag symbols but keep the hashtag words
    # text = re.sub(
    #    r"#",
    #    "",
    #    text
    # )

    # Remove numbers
    text = re.sub(
        r"\d+",
        "",
        text
    )

    # Remove punctuation
    text = text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )

    # Remove extra white spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Split into individual words
    words = text.split()

    # Remove stop words and lemmatize
    words = [

        lemmatizer.lemmatize(word)

        for word in words

        if word not in stop_words

    ]

    # Join words back into a sentence
    cleaned_text = " ".join(words)

    return cleaned_text

In [ ]:
# Apply Preprocessing Function
df["clean_text"] = df["text"].apply(preprocess_text)

In [ ]:
# Display Original + Cleaned Text
print(

    df[
        [
            "text",
            "clean_text"
        ]
    ].head()

)

In [ ]:
# Check For Missing Values
print(

    "\nMissing values in clean_text:",

    df["clean_text"].isnull().sum()

)

### Categorize Ja Gleaner's X Posts by Content

In [ ]:
# Define Category Keywords
categories = {

    "latest news": [
        "breaking", "update", "developing", "incident", "report",
        "police", "statement", "investigation", "emergency", "alert",
        "victim", "scene", "authorities", "response", "confirmed", 
        "unconfirmed", "details", "witness", "operation", "newsflash"
    ]

    "all news": [
        "headline", "coverage", "reporting", "journalism", "editorial", 
        "update", "feature", "newsroom", "brief", "bulletin", 
        "article", "story", "press", "media", "daily",
        "current", "event", "publication", "issue", "summary"
    ]

    "shipping news": [
        "shipping", "cargo", "port", "harbour", "vessel",
        "container", "freight", "logistics", "dock", "marine",
        "customs", "export", "import", "shipment", "terminal",
        "cruise", "sailing", "wharf", "consignment", "berth"
    ],

    "world news": [
        "world", "international", "global", "foreign",
        "country", "nation", "war", "peace", "summit",
        "embassy", "president", "prime minister",
        "treaty", "conflict", "border", "sanctions",
        "diplomacy", "election", "government", "crisis"
    ],

    "sports": [
        "football", "cricket", "netball", "basketball",
        "athletics", "olympics", "match", "coach",
        "goal", "league", "tournament", "medal",
        "championship", "player", "team", "score",
        "race", "victory", "stadium", "competition"
    ],

    "business": [
        "business", "economy", "market", "finance",
        "investment", "profit", "company", "industry",
        "bank", "stock", "trade", "tax",
        "revenue", "inflation", "employment",
        "entrepreneur", "economic", "shareholder",
        "commerce", "budget"
    ],

    "entertainment": [
        "music", "movie", "film", "concert",
        "celebrity", "actor", "actress", "festival",
        "album", "dance", "theatre", "show",
        "television", "tv", "award", "artist",
        "performance", "comedy", "stage", "drama"
    ],

    "commentary": [
        "opinion", "analysis", "editorial", "view",
        "perspective", "commentary", "debate",
        "discussion", "argument", "criticism",
        "reflection", "column", "thoughts",
        "policy", "society", "public", "issue",
        "editor", "comment", "insight"
    ],

    "letters": [
        "letter", "reader", "editor",
        "response", "feedback", "concern",
        "complaint", "suggestion", "request",
        "citizen", "public", "voice",
        "community", "appeal", "message",
        "thank", "question", "reply",
        "correspondence", "submission"
    ],

    "in focus": [
        "focus", "feature", "special",
        "exclusive", "spotlight", "profile",
        "interview", "investigation",
        "documentary", "background",
        "insight", "overview", "deep",
        "series", "coverage", "report",
        "analysis", "highlight", "explore",
        "story"
    ],

    "art & leisure": [
        "art", "gallery", "painting",
        "museum", "culture", "travel",
        "heritage", "craft", "design",
        "photography", "literature",
        "festival", "poetry", "artist",
        "exhibition", "sculpture",
        "creative", "music", "performance",
        "tourism"
    ],

    "food": [
        "food", "recipe", "chef",
        "restaurant", "cook", "cuisine",
        "meal", "kitchen", "nutrition",
        "ingredient", "dining", "breakfast",
        "lunch", "dinner", "dessert",
        "fruit", "vegetable", "drink",
        "menu", "taste"
    ],

    "health": [
        "health", "hospital", "doctor",
        "nurse", "patient", "disease",
        "virus", "medical", "medicine",
        "clinic", "wellness", "mental",
        "exercise", "fitness", "vaccination",
        "treatment", "care", "nutrition",
        "therapy", "healthcare"
    ],

    "auto": [
        "car", "vehicle", "automobile",
        "driver", "road", "traffic",
        "engine", "motor", "truck",
        "SUV", "sedan", "fuel",
        "transport", "tyre", "garage",
        "dealership", "electric vehicle",
        "hybrid", "brake", "highway"
    ]
}

In [ ]:
# Create Categorization Function
def categorize_post(post):

    post = post.lower()

    scores = {}

    for category, keywords in categories.items():

        keyword_count = 0

        for keyword in keywords:

            keyword_count += post.count(keyword)

        scores[category] = keyword_count

    highest_score = max(scores.values())

    if highest_score == 0:
        return "uncategorized"

    return max(scores, key=scores.get)

In [ ]:
# Categorize Each Post
df["category"] = df["clean_text"].apply(categorize_post)

In [ ]:
# Count Post Per Category
category_counts = (
    df["category"]
      .value_counts()
      .sort_index()
)

print(category_counts)

In [ ]:
# Calculate Keyword Match Scores
def keyword_score(post):

    post = post.lower()

    scores = []

    for keywords in categories.values():

        scores.append(
            sum(post.count(word) for word in keywords)
        )

    return max(scores)

df["keyword_matches"] = df["clean_text"].apply(keyword_score)

In [ ]:
# Verify Categorization Results
print(
    df[
        [
            "clean_text",
            "category",
            "keyword_matches"
        ]
    ].head(10)
)

### Measure Category Popularity

In [ ]:
# Calculate Category Statistics
category_statistics = (
    df.groupby("category")["likes"]
      .agg(
          mean_likes="mean",
          median_likes="median",
          number_of_posts="count"
      )
      .round(2)
)

print(category_statistics)

In [ ]:
# Calculate Percentage of Posts Per Category
category_percentages = (
    df["category"]
      .value_counts(normalize=True)
      .mul(100)
      .round(2)
      .rename("percentage")
)

print(category_percentages)

In [ ]:
# Combine Stats Into Table
category_summary = category_statistics.join(category_percentages)

print(category_summary)

In [ ]:
# Sort Categories by Median Likes
category_summary = (
    category_summary
    .sort_values(
        by="median_likes",
        ascending=False
    )
)

print(category_summary)

In [ ]:
# Boxplot of Likes Per Category
plt.figure(figsize=(12,6))

plt.boxplot(
    [
        df[df["category"] == category]["likes"]
        for category in category_summary.index
    ],
    labels=category_summary.index,
    vert=True
)

plt.title("Distribution of Likes by Category")

plt.xlabel("Category")

plt.ylabel("Likes")

plt.xticks(rotation=45)

plt.annotate(
    "Each box shows the spread of likes within a category.",
    xy=(0.35, 1.02),
    xycoords="axes fraction"
)

plt.tight_layout()

plt.show()

In [ ]:
# Histogram of Posts Per Category
category_counts = (
    df["category"]
      .value_counts()
      .sort_index()
)

plt.figure(figsize=(12,6))

plt.bar(
    category_counts.index,
    category_counts.values
)

plt.title("Number of Posts per Category")

plt.xlabel("Category")

plt.ylabel("Number of Posts")

plt.xticks(rotation=45)

plt.annotate(
    "Higher bars indicate categories posted more frequently.",
    xy=(0.35, 1.02),
    xycoords="axes fraction"
)

plt.tight_layout()

plt.show()

In [ ]:
# Histogram of Likes Per Category
plt.figure(figsize=(10,6))

plt.hist(
    df["likes"],
    bins=20
)

plt.title("Distribution of Likes")

plt.xlabel("Likes")

plt.ylabel("Number of Posts")

plt.annotate(
    "Most posts may receive relatively few likes, while a small number receive very high engagement.",
    xy=(0.18, 1.02),
    xycoords="axes fraction"
)

plt.tight_layout()

plt.show()

In [ ]:
# Top Categories by Median Likes
top_categories = (
    category_summary
    .head(5)
)

print(top_categories)

### Engagement Analysis

In [ ]:
# Convert Date Column
df["created_at"] = pd.to_datetime(df["created_at"])

In [ ]:
# Overall Engagement Score
df["engagement"] = (
    df["likes"]
    + df["replies"]
    + df["reposts"]
    + df["quotes"]
)

In [ ]:
# Daily Median Engagement
daily_engagement = (
    df.groupby(
        df["created_at"].dt.date
    )["engagement"]
    .median()
)

In [ ]:
# Engagement Trends Over Time
plt.figure(figsize=(12,6))

plt.plot(
    daily_engagement.index,
    daily_engagement.values,
    marker="o"
)

plt.title("Median Engagement Over Time")

plt.xlabel("Date")

plt.ylabel("Median Engagement")

plt.xticks(rotation=45)

plt.annotate(
    "Median engagement for posts published each day.",
    xy=(0.28, 1.02),
    xycoords="axes fraction"
)

plt.tight_layout()

plt.show()

In [ ]:
# Likes vs Replies
plt.figure(figsize=(8,6))

plt.scatter(
    df["likes"],
    df["replies"]
)

plt.title("Likes vs Replies")

plt.xlabel("Likes")

plt.ylabel("Replies")

plt.annotate(
    "Relationship between likes and replies.",
    xy=(0.45,1.02),
    xycoords="axes fraction"
)

plt.tight_layout()

plt.show()

In [ ]:
# Plot Likes vs Reposts
plt.figure(figsize=(8,6))

plt.scatter(
    df["likes"],
    df["reposts"]
)

plt.title("Likes vs Reposts")

plt.xlabel("Likes")

plt.ylabel("Reposts")

plt.annotate(
    "Relationship between likes and reposts.",
    xy=(0.43,1.02),
    xycoords="axes fraction"
)

plt.tight_layout()

plt.show()

In [ ]:
# Likes vs Quote Posts
plt.figure(figsize=(8,6))

plt.scatter(
    df["likes"],
    df["quotes"]
)

plt.title("Likes vs Quote Posts")

plt.xlabel("Likes")

plt.ylabel("Quote Posts")

plt.annotate(
    "Relationship between likes and quote posts.",
    xy=(0.37,1.02),
    xycoords="axes fraction"
)

plt.tight_layout()

plt.show()

In [ ]:
# Engagement Metrics
engagement_totals = pd.DataFrame({

    "Metric":[
        "Likes",
        "Replies",
        "Reposts",
        "Quote Posts"
    ],

    "Total":[
        df["likes"].sum(),
        df["replies"].sum(),
        df["reposts"].sum(),
        df["quotes"].sum()
    ]
})

print(engagement_totals)

In [ ]:
# Total Engagement
plt.figure(figsize=(8,6))

plt.bar(
    engagement_totals["Metric"],
    engagement_totals["Total"]
)

plt.title("Total Engagement by Metric")

plt.xlabel("Engagement Metric")

plt.ylabel("Total")

plt.annotate(
    "Comparison of overall engagement metrics.",
    xy=(0.32,1.02),
    xycoords="axes fraction"
)

plt.tight_layout()

plt.show()

In [ ]:
# Calculate Median Engagement Per Category
median_category_engagement = (

    df.groupby("category")["engagement"]

    .median()

    .sort_values(
        ascending=False
    )

)

In [ ]:
# Plot Median Engagement Per Category
plt.figure(figsize=(12,6))

plt.bar(
    median_category_engagement.index,
    median_category_engagement.values
)

plt.xticks(rotation=45)

plt.title("Median Engagement by Category")

plt.xlabel("Category")

plt.ylabel("Median Engagement")

plt.annotate(
    "Categories with higher bars generate greater engagement.",
    xy=(0.22,1.02),
    xycoords="axes fraction"
)

plt.tight_layout()

plt.show()

In [ ]:
# Identify Top-Performing Posts
top_posts = (

    df.sort_values(
        by="engagement",
        ascending=False
    )

    .head(10)

)

In [ ]:
# Display Top-Performing Posts
print(

    top_posts[

        [
            "created_at",
            "clean_text",
            "category",
            "engagement"
        ]

    ]

)

In [ ]:
# Visualize Top-Performing Posts
plt.figure(figsize=(12,7))

plt.barh(
    top_posts["clean_text"].str.slice(0,50),
    top_posts["engagement"]
)

plt.title("Top Performing Posts")

plt.xlabel("Total Engagement")

plt.ylabel("Post")

plt.annotate(
    "Posts ranked by total engagement.",
    xy=(0.40,1.02),
    xycoords="axes fraction"
)

plt.tight_layout()

plt.show()

In [ ]:
# Summary Statistics
print(

    df[

        [
            "likes",
            "replies",
            "reposts",
            "quotes",
            "engagement"
        ]

    ].describe()

)

### Content Analysis

In [ ]:
# Count Number of Hastags Per Post
import re

df["hashtag_count"] = df["clean_text"].apply(
    lambda x: len(re.findall(r"#\w+", str(x)))
)

In [ ]:
# Determine Whether Post Has Hastags
df["contains_hashtag"] = (
    df["hashtag_count"] > 0
)

In [ ]:
# Compare Post Engagement With/Without Hastags
hashtag_engagement = (

    df.groupby("contains_hashtag")["engagement"]

    .median()

)

print(hashtag_engagement)

In [ ]:
# Visualize Hashtag Engagement
plt.figure(figsize=(7,5))

plt.bar(
    ["No Hashtags","Hashtags"],
    hashtag_engagement.values
)

plt.title("Median Engagement by Hashtag Usage")

plt.ylabel("Median Engagement")

plt.annotate(
    "Compare median engagement for posts with and without hashtags.",
    xy=(0.18,1.02),
    xycoords="axes fraction"
)

plt.tight_layout()

plt.show()

In [ ]:
# Calculate Post Length
df["post_length"] = (

    df["clean_text"]

    .str.len()

)

In [ ]:
# Visualize Post Length vs Engagement
plt.figure(figsize=(8,6))

plt.scatter(

    df["post_length"],

    df["engagement"]

)

plt.title("Post Length vs Engagement")

plt.xlabel("Post Length (Characters)")

plt.ylabel("Total Engagement")

plt.annotate(

    "Relationship between post length and engagement.",

    xy=(0.28,1.02),

    xycoords="axes fraction"

)

plt.tight_layout()

plt.show()

In [ ]:
# Determine Whether Post Includes Media
df["contains_media"] = (

    df["media_type"]

    .notna()

)

In [ ]:
# Compare Media vs Non-Media Engagement
media_engagement = (

    df.groupby("contains_media")["engagement"]

    .median()

)

print(media_engagement)

In [ ]:
# Visualize Media Engagement
plt.figure(figsize=(7,5))

plt.bar(

    ["No Media","Media"],

    media_engagement.values

)

plt.title("Median Engagement by Media Inclusion")

plt.ylabel("Median Engagement")

plt.annotate(

    "Compare engagement for posts with and without media.",

    xy=(0.20,1.02),

    xycoords="axes fraction"

)

plt.tight_layout()

plt.show()

In [ ]:
# Extract All Hashtags
hashtags = []

for text in df["clean_text"]:

    hashtags.extend(

        re.findall(

            r"#\w+",

            str(text)

        )

    )

In [ ]:
# Find the Most Common Hashtags
from collections import Counter

top_hashtags = Counter(

    hashtags

).most_common(10)

print(top_hashtags)

In [ ]:
# Visualize Top Hashtags
labels = [

    item[0]

    for item in top_hashtags

]

values = [

    item[1]

    for item in top_hashtags

]

plt.figure(figsize=(10,6))

plt.bar(

    labels,

    values

)

plt.xticks(rotation=45)

plt.title("Top 10 Hashtags")

plt.ylabel("Frequency")

plt.annotate(

    "Most frequently used hashtags.",

    xy=(0.34,1.02),

    xycoords="axes fraction"

)

plt.tight_layout()

plt.show()

In [ ]:
# Generate Word Cloud of Post Words
from wordcloud import WordCloud

text = " ".join(

    df["clean_text"]

)

wordcloud = WordCloud(

    width=900,

    height=450,

    background_color="white"

).generate(text)

plt.figure(figsize=(14,7))

plt.imshow(wordcloud)

plt.axis("off")

plt.title("Word Cloud of Ja. Gleaner Company's Posts")

plt.annotate(

    "Larger words appear more frequently.",

    xy=(0.32,1.02),

    xycoords="axes fraction"

)

plt.show()

In [ ]:
# Find Most Common Words in Posts
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(

    stop_words="english"

)

word_matrix = vectorizer.fit_transform(

    df["clean_text"]

)

word_counts = (

    word_matrix.sum(axis=0)

    .A1

)

common_words = pd.DataFrame({

    "Word": vectorizer.get_feature_names_out(),

    "Frequency": word_counts

})

common_words = common_words.sort_values(

    by="Frequency",

    ascending=False

)

print(common_words.head(20))

In [ ]:
# Visualize Most Common Words in Posts
top_words = common_words.head(15)

plt.figure(figsize=(12,6))

plt.bar(

    top_words["Word"],

    top_words["Frequency"]

)

plt.xticks(rotation=45)

plt.title("Most Common Words")

plt.ylabel("Frequency")

plt.annotate(

    "Most frequently occurring words across all posts.",

    xy=(0.22,1.02),

    xycoords="axes fraction"

)

plt.tight_layout()

plt.show()

### Advanced Analysis

#### Sentiment Analysis

In [ ]:
# Import Library
from textblob import TextBlob

In [ ]:
# Create Sentiment Function
def get_sentiment(text):

    polarity = TextBlob(str(text)).sentiment.polarity

    if polarity > 0:

        return "Positive"

    elif polarity < 0:

        return "Negative"

    else:

        return "Neutral"

In [ ]:
# Apply Sentiment Analysis
df["sentiment"] = (

    df["clean_text"]

    .apply(get_sentiment)

)

In [ ]:
# Count Sentiment Categories
sentiment_counts = (

    df["sentiment"]

    .value_counts()

)

print(sentiment_counts)

In [ ]:
# Visualize Sentiment Distribution
plt.figure(figsize=(8,6))

plt.bar(

    sentiment_counts.index,

    sentiment_counts.values

)

plt.title("Distribution of Post Sentiment")

plt.xlabel("Sentiment")

plt.ylabel("Number of Posts")

plt.annotate(

    "Overall sentiment expressed across company posts.",

    xy=(0.28,1.02),

    xycoords="axes fraction"

)

plt.tight_layout()

plt.show()

In [ ]:
# Compare Engagement By Sentiment
sentiment_engagement = (

    df.groupby("sentiment")["engagement"]

    .median()

)

print(sentiment_engagement)

In [ ]:
# Plot Engagement by Sentiment
plt.figure(figsize=(8,6))

plt.bar(

    sentiment_engagement.index,

    sentiment_engagement.values

)

plt.title("Median Engagement by Sentiment")

plt.ylabel("Median Engagement")

plt.annotate(

    "Compare engagement across sentiment categories.",

    xy=(0.25,1.02),

    xycoords="axes fraction"

)

plt.tight_layout()

plt.show()

#### Topic Modeling (LDA)

In [ ]:
# Create Document-Term Matrix
vectorizer = CountVectorizer(

    stop_words="english",

    max_features=1000

)

document_matrix = vectorizer.fit_transform(

    df["clean_text"]

)

In [ ]:
# Train LDA Model
lda = LatentDirichletAllocation(

    n_components=5,

    random_state=42

)

lda.fit(document_matrix)

In [ ]:
# Display Top Words Per Topic
feature_names = vectorizer.get_feature_names_out()

for topic_number, topic in enumerate(lda.components_):

    print(f"\nTopic {topic_number + 1}")

    top_words = [

        feature_names[i]

        for i in topic.argsort()[-10:]

    ]

    print(top_words)

In [ ]:
# Assign Each Post to Topic
topic_probabilities = lda.transform(

    document_matrix

)

df["topic"] = topic_probabilities.argmax(axis=1)

#### K-Means Clustering

In [ ]:
# Import Libraries 
from sklearn.cluster import KMeans

In [ ]:
# Test Different Values of Clusters (K) 

wcss = [] # Store Within-Cluster Sum of Squares (WCSS) a.k.a Inertia

# Test cluster sizes from 1 to 10
cluster_range = range(1, 11)

# Train a K-Means model for each value of k
for k in cluster_range:

    kmeans = KMeans(

        n_clusters=k,

        random_state=42,

        n_init=10

    )

    kmeans.fit(document_matrix)

    # Store the inertia (WCSS)
    wcss.append(kmeans.inertia_)

In [ ]:
# Plot Elbow Curve

plt.figure(figsize=(8, 6))

plt.plot(

    cluster_range,

    wcss,

    marker="o",

    linewidth=2

)

plt.title("Elbow Method for Determining the Optimal Number of Clusters")

plt.xlabel("Number of Clusters (k)")

plt.ylabel("Within-Cluster Sum of Squares (WCSS)")

# Annotate the chart
plt.annotate(

    "Look for the 'elbow' where the decrease in WCSS begins to level off.",

    xy=(0.08, 1.02),

    xycoords="axes fraction"

)

plt.xticks(cluster_range)

plt.grid(True)

plt.tight_layout()

plt.show()

In [ ]:
# Display WCSS Values
for k, inertia in zip(cluster_range, wcss):

    print(f"k = {k}: WCSS = {inertia:.2f}")

In [ ]:
# Train Clustering Model
kmeans = KMeans(

    n_clusters=5,

    random_state=42,

    n_init=10

)

df["cluster"] = kmeans.fit_predict(

    document_matrix

)

In [ ]:
# Count Posts Per Cluster
cluster_counts = (

    df["cluster"]

    .value_counts()

    .sort_index()

)

print(cluster_counts)

In [ ]:
# Visualize Cluster Sizes
plt.figure(figsize=(8,6))

plt.bar(

    cluster_counts.index.astype(str),

    cluster_counts.values

)

plt.title("Number of Posts per Cluster")

plt.xlabel("Cluster")

plt.ylabel("Posts")

plt.annotate(

    "Automatically discovered groups of similar posts.",

    xy=(0.18,1.02),

    xycoords="axes fraction"

)

plt.tight_layout()

plt.show()

In [ ]:
# Examine Example Posts from Each Cluster
for cluster in sorted(df["cluster"].unique()):

    print(f"\nCluster {cluster}")

    print(

        df.loc[
            df["cluster"] == cluster,
            "clean_text"
        ].head(3).tolist()
    )